In [9]:
import torch
from torch.nn import Module, ModuleList, Parameter, Buffer
import tiktoken
import math
import os
import re
import numpy as np
from collections import Counter
print('Hello World')

Hello World


## `Tokenization:` 
#### 1. Splitting a document into unique words:
- split text on whitespace into collection of "words" (but including the whitespace). The following function will convert a text document into a `"corpus"` (a list of unique words split by whitespace) and their corresponding counts. Each word then stored as a list of charaters (`tokens`).
- Example: test target test =>
[[' ', 't', 'e', 's', 't'], [' ', 't', 'a', 'r', 'g', 'e', 't']] =>
[2, 1]

In [10]:
def text_to_corpus(text):
    """
    Convert raw text into unique whitespace-delimited words and counts.

    Input:
        text: str - input text document
    Output:
        tuple(list[list[str]], list[int]) - unique words as token lists and their occurrence counts
    """
    splits = re.split(r'(?=\s)', text) ### split by whitespace, while keeping the whitespace
    words = []
    for word in splits:
        if word is None:
            continue
        words.append(tuple(word)) ### tuple(' text') => (' ','t','e','x','t')
    counts = Counter(words) ### Counter object gives a dictionary: {key:value(count)}
    corpus = []
    count = []
    for key,val in counts.items():
        token = list(key) ### token = collection of characters: [' ','t','e','x','t']
        corpus.append(token) ### corpus = list of tokens: [[' ','t','e','x','t'], [' ', 't', 'a', 'r', 'g', 'e', 't']]
        count.append(val)
    return corpus, count


#### 2. Most common pair
- The following function will compute the most common pair of tokens (collection of characters) in the corpus. This will loop through all pairs of tokens included in each list within the corpus, and return the one with the highest occurance count.

- Example: [[' ','t','e','x','t'], [' ', 't', 'a', 'r', 'g', 'e', 't']] : counts [2,1]

Then this function will return a tuple (' ','t') as this pair occurs three times in the corpus.

In [11]:
def most_common_pair(corpus, counts):
    """
    Find the most frequent adjacent token pair in a weighted corpus.

    Inputs:
        corpus: list[list[str]] - unique words represented as token lists
        counts: list[int] - occurence count for each word in the corpus

    Output:
        tuple(str,str) - most common adjacent token pair, weighted by counts
    """
    pairs = []
    for word, count in zip(corpus,counts):
        for i in range(len(word)-1):
            pair = (word[i],word[i+1]) ### tuple: ()
            pairs.extend([pair] * count)
    count_pair = Counter(pairs) ### gives key(each pair):value(their count)
    return count_pair.most_common(1)[0][0]


#### What's the difference between .extend vs .append ?

In [12]:
lst1 = [1, 2]
lst1.append([3, 4]) ### add the whole list as a singlle element
print(lst1)
lst2 = [1, 2]
lst2.extend([3, 4]) ### let's you add each element from the list.
print(lst2)

[1, 2, [3, 4]]
[1, 2, 3, 4]


In [13]:
words = [[' ','t','e','x','t'], [' ', 't', 'a', 'r', 'g', 'e', 't']]
counts = [2,1]
#print(len(words))
pairs1 = []
for word, count in zip(words,counts):
    for i in range(len(word)-1):
        pair = (word[i], word[i+1])
        #print(pair)
        pairs1.extend([pair]*count)
count_pair = Counter(pairs1)
print(pairs1)
print(count_pair.most_common(2)) #Top two most common pairs based on their counts.
print(count_pair.most_common(1)[0]) #The most common pair based on it's count
print(count_pair.most_common(1)[0][0])

[(' ', 't'), (' ', 't'), ('t', 'e'), ('t', 'e'), ('e', 'x'), ('e', 'x'), ('x', 't'), ('x', 't'), (' ', 't'), ('t', 'a'), ('a', 'r'), ('r', 'g'), ('g', 'e'), ('e', 't')]
[((' ', 't'), 3), (('t', 'e'), 2)]
((' ', 't'), 3)
(' ', 't')


#### 3. Merging a pair
- Now let's write a function that will merge a given pair of tokens together. 
- Example: [' ','t','e','x','t'] => [`' t'`,'e','x','t']

**Note** this function will not return anything, rather it should modify `corpus` ***in place*** (modification directly on corpus). 

In [14]:
def merge_pair(corpus,pair):
    """
    Merge a given token pair everywhere it appears in the corpus.

    Inputs:
        corpus: list[list[str]] - corpus of tokenized words to modify in place
        pair: tuple(str, str) - adjacent token pair to merge
    Output:
        None - modifies corpus directly
    """
    for word in corpus:
        i = 0
        while i < len(word) - 1:
            if (word[i], word[i+1]) == pair:
                merged = word[i] + word[i+1]
                word[i:i+2] = [merged] ### note i+2 as [0:4] => 1st,2nd,3rd not 4th
            else:
                i +=1
                """
                word = ['a', 'b', 'b', 'c']
                pair = ('b', 'b')
                i = 0

                i = 0: ('a', 'b') ≠ pair → else: i = 1
                i = 1: ('b', 'b') = pair → merge → word = ['a', 'bb', 'c'], i stays 1
                i = 1: check again → ('bb', 'c') ≠ pair → else: i = 2
                Loop ends (i < len(word)-1 → 2 < 2 is false)
                """

#### 4. Training a `BPE` (Byte Pair Encoding) Tokenizer
- Finally let's use the above functions (text_to_corpus,most_common_pair,merge_pair) to train a BPE tokenizer, consistent of N different tokens. The basic process for building such a tokenizer is the following:
1. Initialize a base vocabulary of all possible byte-levels. A byte can represent values from: (0-255, total 256 possible values). And each value can be represented as different characters, like: ['\x00', '\x01', ..., 'a', 'b', ..., ' ', ...]. This can be done with the `Python code`: [chr(i) for i in range(256)]
2. Convert the text to corpus and counts, and initialize an empty list of merges
3. Repeat until the number of tokens meets the desired num_tokens:
* Compute the most common pair of tokens
* Add this pair to the list of merges, and add the merged token to the list of tokens (i.e. you'r including it to your vocabulary)
* Merge the pair within the corpus (i.e. you are also updating words according to your vocabulary)

After this, we will have a list of tokens and a list of merges (pair of tokens).


Now, the function should return two intems:

1. The list of tokens, but processed to be returned as a dictionary {`token string`:`token index`} where `token string` is the token itself and `token index` is its index in the list of tokens you originally created. We return things in this manner so that it's faster to encode a sequence of tokens to their indices. The dictionary keys should be in the same order as the original list.
2. The list of merges as you created it (i.e., a list of tuples of pairs of tokens).

In [15]:
def train_bpe(text, num_tokens):
    """
    Train a simple BPE tokenizer from raw text.

    Inputs:
        text: str - training text used to build the tokenizer
        num_tokens: int - total vocabulary size after adding merges
    Output:
        tuple(dic[str, int], list[tuple[str, str]]) - token-to-index mapping and merge list
    """
    tokens = [] #This is the vocabulary you are building to describe any text.
    for i in range(256):
        ### Define tokens as set of 255 different characters
        tokens.append(chr(i)) ### chr(number=97) gives character(e.g. 'a')
    corpus, counts = text_to_corpus(text)
    merges = []
    while len(tokens) < num_tokens: ### as we keep on appending in tokens after we merge
        pair = most_common_pair(corpus,counts)
        merges.append(pair)
        tokens.append(pair[0]+pair[1])
        merge_pair(corpus,pair)
    token_dic = {}
    for idx,tok in enumerate(tokens):
        token_dic[tok]=idx
    return token_dic, merges

#### 5. Encoding and decoding with BPE tokenizer
- To encode text: (i.e. text to token ids)
1. Split the text into a corpus using the same splitting on whitespace as we did before (`re.split(r'(?=\s)', text)`). But convert to a list this time not to tuple.
2. For each pair in merge list, merge that pair in the corpus.
3. After merging all the pais, convert each token into it's numerical id, and return a single (not nested) list of all the tokens in the sequence

In [16]:
def bpe_encode(text, merges, tokens):
    """
    Encode text into token ids using a learned BPE tokenizer.

    Inputs:
        text: str - text to encode
        merges: list[tuple[str,str]] - learned merge operations in order
        tokens: dict[str,int] - mapping from token string to token id
    Output:
        list[int] - encoded token ids for the full text
    """
    splits = re.split(r'(?=\s)', text)
    corpus = []
    for word in splits:
        if not word:
            continue
        corpus.append(list(word)) ### list(' text') => [' ','t','e','x','t']
        ### we need list here because, list is mutable, and so later we can add merged tokens into the corpus.

    ### Adding the trained pairs (from BPE) into your new text corpus
    for pair in merges:
        merge_pair(corpus,pair)

    all_token_ids = []
    for word in corpus:
        for tok in word:
            all_token_ids.append(tokens[tok])
    return all_token_ids


#### To decode text: (i.e. sequence of token ids to text)
*  For each numerical id, find the corresponding token string, and convert join these all together

In [17]:
def bpe_decode(seq, tokens):
    """
    Decode a sequence of BPE token ids back into raw text.

    Input:
        seq: list[int] - token ids to decode (i.e. all_token_ids)
        tokens: dic[str, int] - mapping from token string to token id
    Output:
        str - decode text
    """
    ### Let's reverse the tokens dictionary
    reverse_tokens = {val: key for key, val in tokens.items()}
    text = []
    for ids in seq:
        text.append(reverse_tokens[ids])
    return "".join(text) # text = ["h", "i", "!"] => "".join(text) => "hi!""

### Now let's download the Tiny Stories dataset to tokenize it.

In [18]:
from huggingface_hub import hf_hub_download

repo = "roneneldan/TinyStories"
filename = "TinyStoriesV2-GPT4-train.txt"
if not os.path.exists(filename):
    hf_hub_download(repo_id=repo,filename=filename,repo_type="dataset",local_dir=".")

### Now it's time to Tokenize the data from Tiny Stories
- Since we have implemented all the above required functions, we should be able to build a tokenizer on the Tiny Stories data set using the following code.

In [19]:
with open("TinyStoriesV2-GPT4-train.txt", mode="rt", encoding="latin-1") as f:
    train_text = f.read(100000)
    test_text = f.read(1000)
### Train the BPE with train_text first.
tokens, merges = train_bpe(train_text, 2000)

print("Original text: ", test_text)
tokenized_text = bpe_encode(test_text, merges, tokens) ### text to token ids
print("\nTokens: ", tokenized_text)
assert(test_text == bpe_decode(tokenized_text, tokens)) ### checking if "original text" == "decoded text" ? 
### If assert true nothing happens.

Original text:   deep in the water. One day, he saw a new friend. The new friend was a small green fish named Lily.
Lily was an obedient fish. She always listened to her mom and dad. They told her to be patient when waiting for food. She waited and waited, and then food came.
Bob and Lily liked to play together. They would dive down and then swim up fast. They had lots of fun. They were happy fish friends. And they lived happily in the big blue sea.
<|endoftext|>


Sara loves to play with her kite. She likes to watch it soar high in the sky, like a bird. She holds the leash tight in her hand, so the kite does not fly away.
One day, she goes to the park with her mom and her kite. She sees a big tree with many leaves. She thinks it is a good place to fly her kite. She runs to the tree and lets go of the leash.
"Oh no!" she cries. "My kite is stuck in the tree!"
She tries to pull the leash, but it does not move. She feels sad and angry. She wants her kite back.
"Mom, help me!" she calls. 

## `Conclusion`: Byte Pair Encoding (BPE)

The last goal of BPE (Byte Pair Encoding) is to create a tokenizer that can efficiently represent your text using a fixed-size vocabulary. In simpler terms:

Start with all single characters as tokens.

Iteratively merge the most frequent adjacent pair of tokens, adding each new merged token to your vocabulary.

Repeat until the vocabulary has reached the desired size (num_tokens).

At the end, you have:

A token dictionary {token string → token id}

A list of merges (the order in which pairs were combined)

With this, you can:

Encode text into token IDs for your model

Decode token IDs back to text

✅ Essentially, BPE reduces the number of tokens needed to represent your text, while keeping frequent patterns as single tokens.

### We will use the `tiktoken` library to tokenize the the downloaded data. We load a tokenizer that uses `gpt2` encodings.

### `Why Use the GPT-2 Tokenizer Instead of Our Custom BPE`?

Earlier in this notebook, we implemented a simple Byte Pair Encoding (BPE) tokenizer from scratch to understand how tokenization works internally.

However, for actual LLM training we switch to the GPT-2 tokenizer provided by `tiktoken`:

```python
tokenizer = tiktoken.get_encoding("gpt2")

# Pretokenizing data

### Tokenize a `text file` in chunks and write the token ids to a `binary file`.

Rather than load the text data and tokenize during training, a common optimization is to tokenize the data once, write those tokens to a binary file, and then have our DataLoader class operate on the tokenized representation directly.


In [25]:
def pretokenize_data(tokenizer,in_filename,out_filename,chunk_size=2**20,max_chunks=None):
    """
    Inputs:
        tokenizer : object - tokenizer with an encode() method returning token ids
        in_filename : str - input text filename
        out_filename : str - output binary filename for unit16 tokens
        chunk_size : int - number of text characters to read at a time
        max_chunks : int or None - maximum number of chunks to tokenize
    Output:
        None - writes tokenized data to out_filename
    """
    with open(in_filename,'r') as fin, open(out_filename,'wb') as fout:
        chunk_count = 0
        while True:
            if max_chunks is not None and chunk_count >= max_chunks:
                break
            text = fin.read(chunk_size) ### How many text character to read in each chunk
            if not text:
                break
            tokens = tokenizer.encode(text, allowed_special='all')
            ### convert token ids to binary bytes
            fout.write(np.asarray(tokens, dtype=np.uint16).tobytes()) 
            chunk_count += 1
    return out_filename

In [26]:
tokenizer = tiktoken.get_encoding("gpt2") ### we use this gpt2 tokenizer for encoding
pretokenize_data(tokenizer,
                 "TinyStoriesV2-GPT4-train.txt",
                 "TinyStoriesV2-GPT4-train.small.bin",
                 chunk_size=2**20,
                 max_chunks=2)

'TinyStoriesV2-GPT4-train.small.bin'